# Fuel Use vs Speed: Polynomial Linearization

We model fuel use against speed with a quadratic, fitted with ordinary linear regression:

$$\text{fuel\_use} = w_2 \cdot \text{speed}^2 + w_1 \cdot \text{speed} + b$$

The curve bends, but the model is still linear in the weights. We add a $\text{speed}^2$ column
and fit it with the same least squares method as a straight line.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

from data_loader import DataLoader

## Part 1: Data setup (Nnamdi)

### Source

**Vehicle Energy Dataset (VED)**, https://github.com/gsoh/VED
G. Oh, D. J. LeBlanc, H. Peng, "Vehicle Energy Dataset (VED), A Large-scale Dataset for Vehicle
Energy Consumption Research", *IEEE Transactions on Intelligent Transportation Systems*, 2020.

VED records real-world driving from 383 cars in Ann Arbor, Michigan (Nov 2017 to Nov 2018),
logged through each car's OBD-II port. We use **vehicle 531**, a gasoline car with a 1.5 L
4-cylinder engine, which has the most driving data of any gasoline car in VED: 478 trips.
`src/extract_ved.py` pulls this vehicle's records out of the raw weekly VED files.

In [2]:
df = DataLoader().load()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows: 381,876
Columns: 6


,DayNum,VehId,Trip,Timestamp(ms),Vehicle Speed[km/h],MAF[g/sec]
0,2.519458,531,597,0,35.0,12.24
1,2.519458,531,597,1100,36.0,12.46
2,2.519458,531,597,2100,38.0,12.46
3,2.519458,531,597,3100,38.0,3.36
4,2.519458,531,597,4200,37.0,3.36


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 381876 entries, 0 to 381875
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   DayNum               381876 non-null  float64
 1   VehId                381876 non-null  int64  
 2   Trip                 381876 non-null  int64  
 3   Timestamp(ms)        381876 non-null  int64  
 4   Vehicle Speed[km/h]  381876 non-null  float64
 5   MAF[g/sec]           381876 non-null  float64
dtypes: float64(3), int64(3)
memory usage: 17.5 MB


### Check for missing values and duplicates

In [4]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate (Trip, Timestamp) readings:", df.duplicated(["Trip", "Timestamp(ms)"]).sum())

Missing values per column:
DayNum                 0
VehId                  0
Trip                   0
Timestamp(ms)          0
Vehicle Speed[km/h]    0
MAF[g/sec]             0
dtype: int64

Duplicate rows: 0
Duplicate (Trip, Timestamp) readings: 0


In [5]:
df.describe()

,DayNum,VehId,Trip,Timestamp(ms),Vehicle Speed[km/h],MAF[g/sec]
count,381876.000000,381876.0,381876.000000,3.818760e+05,381876.000000,381876.000000
mean,205.875554,531.0,1894.778114,3.113420e+05,34.089508,8.198270
std,102.562827,0.0,701.446099,2.342659e+05,21.720422,7.622216
min,2.519458,531.0,597.000000,0.000000e+00,0.000000,1.590000
25%,120.495450,531.0,1287.000000,1.211000e+05,17.000000,2.740000
50%,210.803116,531.0,1900.000000,2.653000e+05,36.000000,3.790000
75%,294.653705,531.0,2505.000000,4.569000e+05,51.000000,12.389999
max,375.501195,531.0,3100.000000,1.398200e+06,96.000000,49.709999


No values are missing and there are no duplicate readings. Air flow (MAF) is always positive,
and speeds run from 0 to 96 km/h, so no readings are physically impossible.

### Step 1: Derive fuel rate from air flow

VED does not log `Fuel Rate[L/hr]` for this car. It logs **mass air flow** (MAF), the grams of
air entering the engine per second. A gasoline engine burns fuel at a fixed air-to-fuel mass
ratio of about **14.7 : 1**, and gasoline weighs about **745 g per litre**, so:

$$\text{Fuel Rate [L/hr]} = \frac{\text{MAF [g/s]} \times 3600}{14.7 \times 745}$$

In [6]:
AIR_FUEL_RATIO = 14.7        # grams of air per gram of gasoline
GASOLINE_DENSITY = 745       # grams per litre

analysis_df = df[["Trip", "Timestamp(ms)", "Vehicle Speed[km/h]", "MAF[g/sec]"]].copy()
analysis_df["Fuel Rate[L/hr]"] = (
    analysis_df["MAF[g/sec]"] * 3600 / (AIR_FUEL_RATIO * GASOLINE_DENSITY)
)
analysis_df[["Vehicle Speed[km/h]", "MAF[g/sec]", "Fuel Rate[L/hr]"]].describe()

,Vehicle Speed[km/h],MAF[g/sec],Fuel Rate[L/hr]
count,381876.000000,381876.000000,381876.000000
mean,34.089508,8.198270,2.694952
std,21.720422,7.622216,2.505591
min,0.000000,1.590000,0.522668
25%,17.000000,2.740000,0.900699
50%,36.000000,3.790000,1.245857
75%,51.000000,12.389999,4.072867
max,96.000000,49.709999,16.340775


### Step 2: Remove idling

Readings at 0 km/h are real (the engine still burns fuel while idling), but they describe a
different situation. This experiment studies fuel use as a function of **travelling** speed, and
fuel per distance is undefined when no distance is covered, so we remove them.

In [7]:
rows_loaded = len(analysis_df)
idle = analysis_df["Vehicle Speed[km/h]"] == 0
print(f"Idling readings (speed = 0): {idle.sum():,}")

analysis_df = analysis_df[~idle].copy()
print(f"Rows remaining: {len(analysis_df):,}")

Idling readings (speed = 0): 44,092
Rows remaining: 337,784


### Step 3: Convert to fuel use per distance

The assignment is about **fuel use**, the fuel needed to cover a distance. Dividing the fuel rate
(L/hr) by speed (km/h) gives litres per km, which we scale to L/100 km:

$$\text{Fuel Use [L/100km]} = \frac{\text{Fuel Rate [L/hr]}}{\text{Speed [km/h]}} \times 100$$

In [8]:
analysis_df["Fuel Use[L/100km]"] = (
    analysis_df["Fuel Rate[L/hr]"] / analysis_df["Vehicle Speed[km/h]"] * 100
)
analysis_df["Fuel Use[L/100km]"].describe()

count    337784.000000
mean         11.859080
std          18.738151
min           0.990608
25%           2.702826
50%           6.266265
75%          13.604853
max         704.780185
Name: Fuel Use[L/100km], dtype: float64

### Step 4: Minimum speed

At very low speed the car covers almost no distance, so dividing by speed produces huge L/100 km
values. The table shows how fast this grows below 10 km/h:

In [9]:
low_speed_ranges = pd.cut(analysis_df["Vehicle Speed[km/h]"], [0, 2, 5, 10, 15, 20, 30])
analysis_df.groupby(low_speed_ranges, observed=True)["Fuel Use[L/100km]"].agg(
    ["count", "median", "max"]
).round(1)

,count,median,max
Vehicle Speed[km/h],,,
"(0, 2]",8873,76.3,704.8
"(2, 5]",10091,27.9,242.5
"(5, 10]",13315,14.6,132.1
"(10, 15]",14813,9.2,101.0
"(15, 20]",18638,7.1,77.4
"(20, 30]",48093,5.5,55.8


Below 10 km/h the median fuel use is 15 to 76 L/100 km, with single readings up to 705 L/100 km.
These readings come from crawling in traffic and pulling away from a stop, not from travelling
at a steady speed. Least squares squares every residual, so a handful of them would pull the
curve far off. We keep readings at **10 km/h and above**.

In [10]:
MIN_SPEED = 10  # km/h

rows_before = len(analysis_df)
analysis_df = analysis_df[analysis_df["Vehicle Speed[km/h]"] >= MIN_SPEED].copy()
print(f"Removed below {MIN_SPEED} km/h: {rows_before - len(analysis_df):,}")
print(f"Rows remaining: {len(analysis_df):,}")

Removed below 10 km/h: 29,763
Rows remaining: 308,021


### Step 5: Average fuel use at each speed

Each reading is one second of driving. Its fuel use depends mostly on what the driver was doing
at that moment (accelerating, cruising or coasting), not on speed. Averaging every reading at the
same speed across hundreds of trips cancels those effects out and leaves the effect of speed.

For each whole km/h we compute **total fuel ÷ total distance**, which equals
mean fuel rate ÷ mean speed × 100. Speeds with fewer than 100 readings are too thin to average
reliably and are dropped (that happens above 85 km/h).

In [11]:
MIN_READINGS = 100

speed_df = (
    analysis_df.groupby(analysis_df["Vehicle Speed[km/h]"].round())
    .agg(
        n_readings=("Fuel Rate[L/hr]", "size"),
        mean_rate=("Fuel Rate[L/hr]", "mean"),
        speed=("Vehicle Speed[km/h]", "mean"),
    )
    .reset_index(drop=True)
)
speed_df["fuel_use"] = speed_df["mean_rate"] / speed_df["speed"] * 100
speed_df = speed_df[speed_df["n_readings"] >= MIN_READINGS]
speed_df = speed_df[["speed", "fuel_use", "n_readings"]].reset_index(drop=True)

print(f"Speed points: {len(speed_df)} ({speed_df['speed'].min():.0f} to {speed_df['speed'].max():.0f} km/h)")
print(f"Fewest readings behind one point: {speed_df['n_readings'].min()}")
speed_df.head(10)

Speed points: 76 (10 to 85 km/h)
Fewest readings behind one point: 117


,speed,fuel_use,n_readings
0,10.0,19.684143,2516
1,11.0,17.663447,2696
2,12.0,18.417285,2811
3,13.0,16.782284,2949
4,14.0,15.566125,3152
5,15.0,15.868439,3205
6,16.0,14.450312,3378
7,17.0,14.920329,3540
8,18.0,14.482670,3732
9,19.0,13.701966,3802


### Cleaning summary

In [12]:
pd.DataFrame(
    {
        "step": [
            f"Loaded (vehicle 531, {df['Trip'].nunique()} trips)",
            "After removing idling (speed = 0)",
            f"After removing speed below {MIN_SPEED} km/h",
            f"Averaged per km/h (speeds with >= {MIN_READINGS} readings)",
        ],
        "rows_remaining": [rows_loaded, rows_before, len(analysis_df), len(speed_df)],
    }
)

,step,rows_remaining
0,"Loaded (vehicle 531, 478 trips)",381876
1,After removing idling (speed = 0),337784
2,After removing speed below 10 km/h,308021
3,Averaged per km/h (speeds with >= 100 readings),76


### Handoff

| Variable | What it is | Used by |
|---|---|---|
| `speed_df` | One row per km/h: `speed` (km/h), `fuel_use` (L/100 km), `n_readings` | Everyone: this is the modelling data |
| `analysis_df` | Cleaned per-second readings, speed ≥ 10 km/h | Rangeetha, to show the raw scatter |

All models are fitted on `speed_df`, so `x = speed_df["speed"]` and `y = speed_df["fuel_use"]`.

## Part 2: Transformation + exploratory analysis (Rangeetha)

## Part 3: NumPy model (Davis)

## Part 4: scikit-learn + model checking (Carlos)